In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

# Setup
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Step 1: Go to the job listing page
driver.get("https://jobs.myflorida.com/search/")
time.sleep(5)  # Wait for page to fully load

# Step 2: Collect all unique job URLs
job_elements = driver.find_elements("css selector", "a.jobTitle-link")
job_urls = []
for elem in job_elements:
    url = elem.get_attribute("href")
    if url and url not in job_urls:
        job_urls.append(url)

print(len(job_urls))
# Step 3: Open each URL in a new tab exactly once
for url in job_urls:
    driver.execute_script(f"window.open('{url}', '_blank');")
    time.sleep(1)  # slight delay to avoid overloading the browser

# Step 4: Optional — wait before closing
time.sleep(15)
driver.quit()


25


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

def scrape_job_page(url):
    # Setup headless browser
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    try:
        driver.get(url)
        time.sleep(3)  # Wait for content to load

        # Scrape full page text content
        body_text = driver.find_element("tag name", "body").text
        return body_text
    
    finally:
        driver.quit()


In [3]:
scrape_job_page(job_urls[1])

'View Profile\nSearch by Postal Code\nShow More Options\nClear\nSelect how often (in days) to receive an alert: Create Alert\nApply Now »\nCRIME LABORATORY POSITIONS- 71001626 1\nDate:  Jan 4, 2026\n\nThe State Personnel System is an E-Verify employer. For more information click on our E-Verify Website.\nRequisition No: 862735 \nAgency: Department of Law Enforcement\nWorking Title: CRIME LABORATORY POSITIONS- 71001626 1\n Pay Plan: Career Service\nPosition Number: 71001626 \nSalary:  $42,200.08 - $56,732.52 \nPosting Closing Date: 03/31/2026 \nTotal Compensation Estimator Tool\nFORENSIC LABORATORY POOL\nCRIME LABORATORY ANALYSTS & FORENSIC TECHNOLOGISTS\n**Open-Competitive Opportunity**\n  POSITION SUMMARY:\nThe Florida Department of Law Enforcement (FDLE) is currently accepting applications for passionate and motivated applicants looking for a challenging opportunity and interested in becoming part of FDLE forensics. Our casework strategy requires a team dynamic to successfully serve 

In [4]:
import json
import re

def extract_job_data(raw_text, job_link):
    # Extract job title
    title_match = re.search(r"\n(.+?)\s*-\s*\d{8}\nDate:", raw_text)
    job_title = title_match.group(1).strip() if title_match else ""

    # Extract job ID
    job_id_match = re.search(r"Working Title: .*? - (\d{8})", raw_text)
    job_id = job_id_match.group(1) if job_id_match else ""

    # Extract posting date
    date_match = re.search(r"Date:\s+(.*?\d{4})", raw_text)
    posting_date = date_match.group(1).strip() if date_match else ""

    # Extract salary
    salary_match = re.search(r"Salary:\s+(.*?)\n", raw_text)
    salary = salary_match.group(1).strip() if salary_match else ""

    # Extract agency
    agency_match = re.search(r"Agency:\s+(.*)", raw_text)
    agency = agency_match.group(1).strip() if agency_match else ""

    # Extract location
    loc_match = re.search(r"Location:\s+(.*)", raw_text)
    location = loc_match.group(1).strip() if loc_match else ""

    # Extract main job description from first major paragraph after the title
    job_desc_start = raw_text.find("*****OPEN COMPETITIVE")  # or another clear marker
    job_desc = raw_text[job_desc_start:].strip() if job_desc_start != -1 else raw_text.strip()

    # Prepare dictionary
    job_data = {
        "job_link": job_link,
        "job_id": job_id,
        "job_title": job_title,
        "posting_date": posting_date,
        "salary": salary,
        "agency": agency,
        "location": location,
        "job_description": job_desc
    }

    return job_data

def save_to_json(data, filename="job_data.json"):
    with open(filename, "a") as f:
        json.dump(data, f, indent=2)
        f.write(",\n")  # Append mode so multiple jobs can be saved


In [5]:
extract_job_data(scrape_job_page(job_urls[1]),job_urls[1])

{'job_link': 'https://jobs.myflorida.com/job/FORT-MYERS-CRIME-LABORATORY-POSITIONS-71001626-1-FL-33902/1333444700/',
 'job_id': '',
 'job_title': '',
 'posting_date': 'Jan 4, 2026',
 'salary': '$42,200.08 - $56,732.52',
 'agency': 'Department of Law Enforcement',
 'location': 'FORT MYERS, FL, US, 33902 JACKSONVILLE, FL, US, 32256 TALLAHASSEE, FL, US, 32309 TAMPA, FL, US, 34748 ORLANDO, FL, US, 32825 FT MYERS, FL, US, 33905 TALLAHASSEE, FL, US, 32317 TALLAHASSEE, FL, US, 32308 ORLANDO, FL, US, 32835 JACKSONVILLE, FL, US, 33207 PENSACOLA, FL, US, 32505 ORLANDO, FL, US, 32811 ORLANDO, FL, US, 32804 JACKSONVILLE, FL, US, 32207 JACKSONVILLE, FL, US, 32208 TAMPA, FL, US, 33634 FORT MYERS, FL, US, 33908 FORT MYERS, FL, US, 33912 TAMPA, FL, US, 33612 JACKSONVILLE, FL, US, 32250 ORLANDO, FL, US, 32824 JACKSONVILLE, FL, US, 32204 JACKSONVILLE, FL, US, 32206 JACKSONVILLE, FL, US, 32222 JACKSONVILLE, FL, US, 32231 TAMPA, FL, US, 33602 ORLANDO, FL, US, 32801 TAMPA, FL, US, 33605 FT MYERS, FL, US, 3

In [6]:
from sentence_transformers import SentenceTransformer, util

def compute_job_match(jd_text, keyword_list):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Create embeddings
    jd_embedding = model.encode(jd_text, convert_to_tensor=True)
    keyword_embeddings = model.encode(keyword_list, convert_to_tensor=True)
    
    # Compute cosine similarities
    similarities = util.cos_sim(jd_embedding, keyword_embeddings)[0]
    average_score = float(similarities.mean())
    
    # Optional: detailed similarity per keyword
    keyword_scores = {kw: float(score) for kw, score in zip(keyword_list, similarities)}
    
    return {
        "match_percentage": round(average_score * 100, 2),
        "keyword_scores": keyword_scores
    }


In [17]:
jd = (
    "DESCRIPTION: This position reports to the Enterprise Data Management (EDM) Unit Manager and will play an integral part of the EDM "
    "team, whose functions support current and future Office of Information Technology activities, specifically enterprise-wide reporting, "
    "enterprise-wide data analytics, data exchange, master client management, and technical assistance to all DCF stakeholders. This position "
    "is a junior level Operations review specialist and requires some skills in SQL programming and a suite of business intelligence tools. "
    "EXAMPLES OF WORK PERFORMED: (Note: The omission of specific statements does not preclude management from assigning "
    "specific duties not listed herein if such duties are a logical assignment to the position.) "
    "Develop and validate old and new EDM dashboard and reports based on customer business requirements gathered. "
    "Working with senior team members to create extracts and other implementation strategies in accordance with federal, state, and "
    "departmental objectives. Specifically for reports developed inside/outside EDM team. "
    "Maintain documentation for all dashboards and reports developed. "
    "Attending meetings and workgroups with other team members to gather business requirements for future deliverables. "
    "Performing other related duties. "
    "KNOWLEDGE, SKILLS AND ABILITIES: "
    "Experience and knowledge in methods of data collection and analysis. "
    "Knowledge of basic SDLC practices. "
    "Ability to evaluate and analyze data. "
    "Solve problems and document workflows and other activities to support leadership. "
    "Ability to communicate directly. "
    "Experience with advanced features of MS excel. "
    "Ability to interpret and communicate complex programming requirements. "
    "Some knowledge of Power BI, Qlik and other business intelligence tools. "
    "Ability to communicate and interpret the needs of customers."
)

keywords = ["data analysis", "Power BI", "SQL", "forecasting", "ETL"]

result = compute_job_match(jd, keywords)
print(result)


{'match_percentage': 22.85, 'keyword_scores': {'data analysis': 0.31146714091300964, 'Power BI': 0.10062478482723236, 'SQL': 0.2962341904640198, 'forecasting': 0.11748114973306656, 'ETL': 0.31648939847946167}}


In [18]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import re

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample job description
job_description = """
This position reports to the Enterprise Data Management (EDM) Unit Manager and will play an integral part of the EDM
team, whose functions support current and future Office of Information Technology activities, specifically enterprise-wide reporting,
enterprise-wide data analytics, data exchange, master client management, and technical assistance to all DCF stakeholders. This position
is a junior level Operations review specialist and requires some skills in SQL programming and a suite of business intelligence tools.

Develop and validate old and new EDM dashboard and reports based on customer business requirements gathered.
Working with senior team members to create extracts and other implementation strategies in accordance with federal, state, and
departmental objectives. Specifically for reports developed inside/outside EDM team.
Maintain documentation for all dashboards and reports developed.
Attending meetings and workgroups with other team members to gather business requirements for future deliverables.
Performing other related duties.

Experience and knowledge in methods of data collection and analysis.
Knowledge of basic SDLC practices.
Ability to evaluate and analyze data.
Solve problems and document workflows and other activities to support leadership.
Ability to communicate directly.
Experience with advanced features of MS excel.
Ability to interpret and communicate complex programming requirements.
Some knowledge of Power BI, Qlik and other business intelligence tools.
Ability to communicate and interpret the needs of customers.
"""

# Enhanced keyword list (with context)
keywords = [
    "building dashboards with Power BI",
    "writing SQL queries for reporting",
    "designing ETL pipelines",
    "data visualization",
    "business reporting",
    "collaborating with stakeholders"
]

# Embed job description and keywords
job_embedding = model.encode(job_description, convert_to_tensor=True)
keyword_embeddings = model.encode(keywords, convert_to_tensor=True)

# Calculate cosine similarities
cosine_scores = util.cos_sim(job_embedding, keyword_embeddings)[0].cpu().numpy()

# Basic frequency boost
freq_boost = []
for kw in keywords:
    raw_kw = re.sub(r"[^\w\s]", "", kw.lower())
    count = job_description.lower().count(raw_kw)
    freq_boost.append(min(count * 0.05, 0.15))  # Cap boost

# Hybrid score: weighted combo of semantic + frequency presence
hybrid_scores = 0.85 * cosine_scores + 0.15 * np.array(freq_boost)

# Compute average match
match_percentage = float(np.mean(hybrid_scores)) * 100

# Detailed output
keyword_score_dict = {kw: float(score) for kw, score in zip(keywords, hybrid_scores)}

{
    "match_percentage": round(match_percentage, 2),
    "keyword_scores": keyword_score_dict
}


{'match_percentage': 25.26,
 'keyword_scores': {'building dashboards with Power BI': 0.16463343799114227,
  'writing SQL queries for reporting': 0.28077635169029236,
  'designing ETL pipelines': 0.2804112434387207,
  'data visualization': 0.20095877349376678,
  'business reporting': 0.37417539954185486,
  'collaborating with stakeholders': 0.21464161574840546}}

In [21]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [24]:
import ollama
import json

def ollama_semantic_match(jd_text, skills_list, model='mistral'):
    prompt = f"""
You are a helpful assistant skilled in understanding job descriptions.

Here is the job description:
\"\"\"{jd_text}\"\"\"

Now, I will give you a list of key skills or qualifications.

Your task is to:
1. Rate how well the job description matches each skill on a scale from 0 to 1.
2. Calculate an overall match percentage as an average of all scores.
3. Return the results as a JSON with the structure:
{{"match_percentage": <value>, "keyword_scores": {{"skill_1": <score>, ...}} }}

Here are the skills: {skills_list}
"""
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    try:
        result_text = response['message']['content']
        return json.loads(result_text)
    except Exception as e:
        print("Model response couldn't be parsed as JSON:")
        print(response['message']['content'])
        return None


In [26]:
#jd = "This position requires experience in building dashboards using Power BI, writing SQL queries, and working with ETL tools"


jd = """
This position reports to the Enterprise Data Management (EDM) Unit Manager and will play an integral part of the EDM
team, whose functions support current and future Office of Information Technology activities, specifically enterprise-wide reporting,
enterprise-wide data analytics, data exchange, master client management, and technical assistance to all DCF stakeholders. This position
is a junior level Operations review specialist and requires some skills in SQL programming and a suite of business intelligence tools.

Develop and validate old and new EDM dashboard and reports based on customer business requirements gathered.
Working with senior team members to create extracts and other implementation strategies in accordance with federal, state, and
departmental objectives. Specifically for reports developed inside/outside EDM team.
Maintain documentation for all dashboards and reports developed.
Attending meetings and workgroups with other team members to gather business requirements for future deliverables.
Performing other related duties.

Experience and knowledge in methods of data collection and analysis.
Knowledge of basic SDLC practices.
Ability to evaluate and analyze data.
Solve problems and document workflows and other activities to support leadership.
Ability to communicate directly.
Experience with advanced features of MS excel.
Ability to interpret and communicate complex programming requirements.
Some knowledge of Power BI, Qlik and other business intelligence tools.
Ability to communicate and interpret the needs of customers.
"""
skills = [
    "building dashboards with Power BI",
    "writing SQL queries for reporting",
    "designing ETL pipelines",
    "data visualization",
    "business reporting",
    "collaborating with stakeholders"
]

result = ollama_semantic_match(jd, skills)
print(json.dumps(result, indent=2))


Model response couldn't be parsed as JSON:
 Here is how each skill matches the job description based on the provided text, rated from 0 to 1:

1. Building dashboards with Power BI: 0.9
2. Writing SQL queries for reporting: 0.8
3. Designing ETL pipelines: 0.4 (There is no mention of designing ETL pipelines in the job description, but some extraction strategies are mentioned)
4. Data visualization: 1
5. Business reporting: 1
6. Collaborating with stakeholders: 0.8

Now let's calculate the overall match percentage as an average of all scores:
(0.9 + 0.8 + 0.4 + 1 + 1 + 0.8) / 6 = 0.917 or 91.7%

Finally, here is the JSON structure with the results:

{"match_percentage": 0.917, "keyword_scores": {"building dashboards with Power BI": 0.9, "writing SQL queries for reporting": 0.8, "designing ETL pipelines": 0.4, "data visualization": 1, "business reporting": 1, "collaborating with stakeholders": 0.8} }
null
